# 07 · Evaluation, baselines & ablations
All numbers reported with bootstrap 95% CIs over ≥ 3 seeds. The primary, pre-registered metric is **under-triage rate** on the human-written test set.

| Section | What it measures |
|---|---|
| A | Rule baseline vs. BioBERT per register (the euphemism gap) |
| B | Ablations: alignment, KG, conformal, evidential fusion, OOD gate, planner cost term |
| C | Adaptive questioning: questions & disclosure cost to decision |
| D | Zero-shot LLM baseline (calibration & under-triage) |

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
# A. Rule baseline (no trained models) by register
!python -m eval.run_text_eval --data data/stigmasymp_w/val.jsonl --out reports/text_eval_rule_val.json
# Reportable: human-written test set
# !python -m eval.run_text_eval --data data/human_test.jsonl --out reports/text_eval_rule_test.json

In [ ]:
# A. Trained configuration (fill checkpoint ids in configs/trained.yaml first)
# !python -m eval.run_text_eval --data data/human_test.jsonl --config configs/trained.yaml --out reports/text_eval_trained_test.json

## C. Adaptive questioning: simulated patients

In [ ]:
import random

import numpy as np

from rarecare.agent.reasoner import QuestionPlanner, TierReasoner, TriageState
from rarecare.config import TriageConfig
from rarecare.kg.graph import default_graph

kg = default_graph(); reasoner = TierReasoner(kg)

def simulate(cost_exponent, n=500, seed=13):
    rng = random.Random(seed); planner = QuestionPlanner(reasoner, TriageConfig(cost_exponent=cost_exponent, max_questions=6))
    concepts = sorted(kg.concepts); asked, cost, under = [], [], 0
    for _ in range(n):
        present = set(rng.sample(concepts, rng.choice([1, 2])))
        truth = {v: rng.choices(kg.variables[v].values, kg.variables[v].prior)[0] for v in kg.variables}
        true_tier, _, _ = reasoner.evaluate(present, truth)
        st, c = TriageState(present=present), 0
        while True:
            d = planner.decide(st)
            if d.question is None: break
            st.variables[d.question.variable] = truth[d.question.variable]; st.questions_asked += 1; c += d.question.disclosure_cost
        asked.append(st.questions_asked); cost.append(c); under += d.tier.severity < true_tier.severity
    return {"mean_questions": np.mean(asked), "mean_disclosure_cost": np.mean(cost), "under_triage": under / n}

{"EIG only (cost ignored)": simulate(0.0), "EIG / cost": simulate(1.0), "EIG / cost^2": simulate(2.0)}

## Results table (filled after training, kept honest)
| Model | Register | Concept F1 | Under-triage ↓ | Urgent recall | ECE ↓ |
|---|---|---|---|---|---|
| Rule baseline | clinical / lay / euph. / Hinglish | – | – | – | – |
| BioBERT | … | – | – | – | – |
| BioBERT + SapBERT-aligned | … | – | – | – | – |
| Zero-shot LLM | … | – | – | – | – |